In [1]:
!pip install openai pandas numpy

In [2]:
from google.colab import userdata
key = userdata.get('openai_key')

In [3]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [4]:
import openai
import pandas as pd
from pandas import DataFrame
import json
import random
import numpy as np
openai.api_key = key


In [5]:
def get_gpt_response(messages:list, anchor:str, text_a:str, text_b:str):
    response = openai.chat.completions.create(
        temperature=0.5,
        top_p=0.9,
        model="gpt-3.5-turbo",
        messages=messages,
        timeout = 400
    )
    return response.choices[0].message.content.strip()

In [8]:
anchor=''
text_a=''
text_b=''

In [6]:
system_prompt= """
You are an expert in text analysis.


Tell me if A has similar abstract theme to Achor.
Tell me if B has similar abstract theme to Achor.

Return 'A' or 'B'

Return only in one character, "A" or "B" , with no explaination
"""

In [9]:
message = [{"role": "system", "content": system_prompt}, {"role": "user", "content": f"Anchor: {anchor} A: {text_a} B: {text_b}, answer with no explaination"}]





In [10]:
messages_switched = [{"role": "system", "content": system_prompt}, {"role": "user", "content": f"Anchor: {anchor} B: {text_b} A: {text_a}, answer with no explaination"}]

In [11]:
input = pd.read_json("gdrive/MyDrive/SemEval26/dev_track_a.jsonl", lines=True)
#input_array = np.array_split(input, 4)
answers = []

#for arr in input_array:
for i in range(len(input)):
  anchor, text_a, text_b = input['anchor_text'][i], input['text_a'][i], input['text_b'][i]
  response = get_gpt_response(message,anchor,text_a,text_b)
  answers.append(response)



In [12]:
answers_switched = []

#for arr in input_array:
for i in range(len(input)):
  anchor, text_a, text_b = input['anchor_text'][i], input['text_a'][i], input['text_b'][i]
  response_switched = get_gpt_response(messages_switched,anchor,text_a,text_b)
  answers_switched.append(response_switched)


In [13]:

  len(answers)
  print(answers_switched)

['B', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'B', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'B', 'A', 'B', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'B', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'B', 'A', 'A', 'A', 'A', 'A', 'B', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'B', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'B', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'B', 'A', 'A', 'B', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'B', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'B']

In [16]:
def a_is_closer(answer):
  A_is_closer=[]
  for i in range(len(answer)):
    if answer[i] == 'A':
      A_is_closer.append(True)
    else:
      A_is_closer.append(False)


  return A_is_closer

In [17]:

input['predicted_A_is_closer'] = a_is_closer(answers)

accuracy = (input["predicted_A_is_closer"] == input["text_a_is_closer"]).mean()
accuracy

np.float64(0.515)

In [18]:
input['predicted_A_is_closer_switched'] = a_is_closer(answers_switched)
accuracy = (input["predicted_A_is_closer_switched"] == input["text_a_is_closer"]).mean()
accuracy

np.float64(0.5)

In [19]:
difference = (input["predicted_A_is_closer_switched"] == input["predicted_A_is_closer"]).mean()
difference

np.float64(0.465)